In [3]:
import pandas as pd

In [5]:
df = pd.read_csv("creditcard.csv")

In [6]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
df.shape
df['Class'].value_counts()
df.isnull().sum()


Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64

In [8]:
df = df.sort_values('Time')

split_index = int(0.8 * len(df))

train = df.iloc[:split_index]
test  = df.iloc[split_index:]


In [9]:
print(train['Class'].value_counts())
print(test['Class'].value_counts())


Class
0    227428
1       417
Name: count, dtype: int64
Class
0    56887
1       75
Name: count, dtype: int64


In [10]:
X_train = train.drop('Class', axis=1)
y_train = train['Class']

X_test = test.drop('Class', axis=1)
y_test = test['Class']


In [11]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(227845, 30) (227845,)
(56962, 30) (56962,)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount']  = scaler.transform(X_test[['Amount']])


In [13]:
"""from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=577,   # VERY important
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, random_state=42, ...)

In [14]:
"""y_probs = xgb_model.predict_proba(X_test)[:, 1]


In [15]:
"""from sklearn.metrics import average_precision_score

print("PR-AUC:", average_precision_score(y_test, y_probs))


PR-AUC: 0.8027112442243945


In [16]:
"""from sklearn.metrics import classification_report

for t in [0.005, 0.01, 0.02, 0.05]:
    print(f"\nThreshold: {t}")
    y_pred = (y_probs > t).astype(int)
    print(classification_report(y_test, y_pred))



Threshold: 0.005
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56887
           1       0.06      0.89      0.12        75

    accuracy                           0.98     56962
   macro avg       0.53      0.94      0.55     56962
weighted avg       1.00      0.98      0.99     56962


Threshold: 0.01
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     56887
           1       0.11      0.87      0.19        75

    accuracy                           0.99     56962
   macro avg       0.55      0.93      0.59     56962
weighted avg       1.00      0.99      0.99     56962


Threshold: 0.02
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56887
           1       0.19      0.84      0.32        75

    accuracy                           1.00     56962
   macro avg       0.60      0.92      0.66     56962
weighted avg       1.0

In [17]:
from sklearn.tree import DecisionTreeClassifier


In [18]:
dt_model = DecisionTreeClassifier(
    max_depth=6,                 # VERY important
    min_samples_split=50,
    min_samples_leaf=50,
    class_weight={0: 1, 1: 577}, # handle imbalance
    random_state=42
)


In [19]:
dt_model.fit(X_train, y_train)


DecisionTreeClassifier(class_weight={0: 1, 1: 577}, max_depth=6,
                       min_samples_leaf=50, min_samples_split=50,
                       random_state=42)

In [20]:
y_probs_dt = dt_model.predict_proba(X_test)[:, 1]


In [21]:
y_pred_dt = (y_probs_dt > 0.02).astype(int)


In [22]:
from sklearn.metrics import classification_report, average_precision_score

print("PR-AUC:", average_precision_score(y_test, y_probs_dt))
print(classification_report(y_test, y_pred_dt))


PR-AUC: 0.6915235849965888
              precision    recall  f1-score   support

           0       1.00      0.76      0.86     56887
           1       0.00      0.85      0.01        75

    accuracy                           0.76     56962
   macro avg       0.50      0.81      0.44     56962
weighted avg       1.00      0.76      0.86     56962

